In [ ]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

sys.path.append("../src") 
from validate_adv import invalid_reason, edit_distance, ext
from evaluate import evaluate, adversarial_metrics, save_preds

ADV_DIR = Path("../data/adversarial")
ADV_DIR.mkdir(parents=True, exist_ok=True)
Path("../results").mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


# Configuration

In [ ]:
with open('../data/vocab.json') as f:
    vocab = json.load(f)    
    index_to_string = vocab["index_to_string"]

with open('../data/config.json') as f:
    config = json.load(f)

char_to_idx = {c: i for i, c in enumerate(vocab['index_to_string'])}
PAD_IDX = vocab['pad_idx']
UNK_IDX = vocab['unk_idx']
MAX_LEN = config['max_len']
VOCAB_SIZE = len(index_to_string)

print(f'vocab={len(char_to_idx)}, MAX_LEN={MAX_LEN}, PAD_IDX={PAD_IDX}, UNK_IDX={UNK_IDX}')

# Attack config
K = 3 #edit budget
TOP_POSITIONS = 3 #positions tried each round
THRESHOLD = 0.5 #p_phish >= 0.5 --> phishing class


train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

print("train:", train_df.shape, "| test:", test_df.shape)



vocab=67, MAX_LEN=256, PAD_IDX=0, UNK_IDX=1
train: (185397, 2) | test: (23175, 2)


# Load CNN Baseline (MODEL 1)

In [19]:
class CharacterCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.conv1 = nn.Conv1d(embedding_dim, 128, kernel_size=5)
        self.conv2 = nn.Conv1d(128, 128, kernel_size=5)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x).squeeze(2)
        return self.fc(x)


model = CharacterCNN(VOCAB_SIZE, embedding_dim=32).to(device)
model.load_state_dict(torch.load("../models/cnn_baseline.pt", map_location=device, weights_only=True))
model.eval()
for p in model.parameters():
    p.requires_grad_(False)  # we only need gradients w.r.t. the embeddings, not the weights
print("MODEL 1 loaded.")

MODEL 1 loaded.


# Helpers 
- returns P(phishing)

In [34]:
def encode_batch(urls):
    arr = np.full((len(urls), MAX_LEN), PAD_IDX, dtype=np.int64)
    for r, u in enumerate(urls):
        row = [char_to_idx.get(c, UNK_IDX) for c in u[:MAX_LEN]]
        arr[r, :len(row)] = row
    return torch.from_numpy(arr) #(batch size, max_len)

@torch.no_grad()
def p_phish_batch(urls, batch_size=1024):
    # predict phishing probability
    out = []
    for i in range(0, len(urls), batch_size):
        x = encode_batch(urls[i:i + batch_size]).to(device)
        out.append(torch.sigmoid(model(x).squeeze(1)).cpu())
    return torch.cat(out).numpy() 

def forward_from_embeddings(emb):
    # The CNN forward pass, starting from the embedding output (same layers/order as CharacterCNN.forward)
    x = emb.transpose(1, 2)
    x = model.relu(model.conv1(x))
    x = model.relu(model.conv2(x))
    x = model.pool(x).squeeze(2)
    return model.fc(x)

def logit_and_embedding_grad(url):
    # Gradient of the phishing logit w.r.t. the embedding of every character position.
    # gradient shows how much each character position is pushing the model toward “phishing”
    ids = encode_batch([url]).to(device)
     # inference_mode(False) + enable_grad(): make sure autograd is on even if the notebook is in no_grad / inference mode
    with torch.inference_mode(False), torch.enable_grad():
        emb = model.embedding(ids).detach().clone().requires_grad_(True)   # leaf tensor we differentiate against
        logit = forward_from_embeddings(emb).squeeze()
        (grad,) = torch.autograd.grad(logit, emb)
    return ids[0], grad[0]  # (L,), (L, D)

# the explicit forward must reproduce model(ids), otherwise the gradients are for a different function
with torch.no_grad():
    _ids = encode_batch(["login.example.com/a?b=1"]).to(device)
    assert torch.allclose(forward_from_embeddings(model.embedding(_ids)), model(_ids), atol=1e-5), "forward mismatch"
print("gradient forward matches model()")

gradient forward matches model()


# Permissible edits
- One edit = substitute, insert or delete one character.
- only subdomain, path, query, fragment are editable

In [35]:
HOST_CHARS = "abcdefghijklmnopqrstuvwxyz0123456789-."
PATH_CHARS = "abcdefghijklmnopqrstuvwxyz0123456789-._~/=&?#"
START_CHARS = "/?#"  # the only single-char extensions of a URL that has no path yet (e.g. ph.gp -> ph.gp/)

assert all(c in char_to_idx for c in set(HOST_CHARS + PATH_CHARS))


def char_mask(chars):
    m = torch.zeros(VOCAB_SIZE, dtype=torch.bool, device=device)
    m[[char_to_idx[c] for c in chars]] = True
    return m


REGION_MASKS = torch.stack([char_mask(HOST_CHARS), char_mask(PATH_CHARS), char_mask(START_CHARS)])  # 0=host 1=path 2=start


def url_layout(url):
    # find editable parts of current url
    # Returns (n, path_start, host_range). host_range = (start, end) of the subdomain, or None if there is none
    n = len(url)
    path_start = next((i for i, c in enumerate(url) if c in "/?#"), n)  # end of authority
    userinfo, at, hostport = url[:path_start].rpartition("@")
    host_start = len(userinfo) + len(at)
    host = hostport.split(":")[0]
    sub = ext(host).subdomain if host else ""
    if not sub or not host.startswith(sub):
        return n, path_start, None
    return n, path_start, (host_start, host_start + len(sub))

def ins_chars(j, layout):
    # Characters allowed for an insertion *before* index j (None if not allowed there)
    n, ps, hr = layout # length, path start, host range
    if hr and hr[0] <= j <= hr[1]:
        return HOST_CHARS
    if n > ps and ps + 1 <= j <= n:
        return PATH_CHARS
    if n == ps and j == n:
        return START_CHARS
    return None


# Gradient scoring and candidates

- replacing char `a` at position `i` with char `b` changes the phishing logit by about `grad_i · (E[b] − E[a])` 
- A position's score is the best (most negative) estimate over all allowed `b`.


In [36]:
E = model.embedding.weight.detach()  # (V, D)

def top_positions(url, layout, m=TOP_POSITIONS):
    #pick which position in url to try editing using first order est. (finding best positions that could reduce P(phish))
    n, ps, hr = layout
    ids, grad = logit_and_embedding_grad(url)

    pos, region = [], []
    if hr:
        for i in range(hr[0], min(hr[1], MAX_LEN)):
            pos.append(i); region.append(0)
    for i in range(ps + 1, min(n, MAX_LEN)):
        pos.append(i); region.append(1)
    if n < MAX_LEN:  # append position (also covers "start a path" for domain-only URLs)
        pos.append(n); region.append(2 if n == ps else 1)
    if not pos:
        return []

    pos_t = torch.tensor(pos, device=device)
    allowed = REGION_MASKS[torch.tensor(region, device=device)].clone()      # (P, V)
    allowed[torch.arange(len(pos), device=device), ids[pos_t]] = False       # b != current char

    gain = -(grad @ E.T) + (grad * E[ids]).sum(1, keepdim=True)              # predicted logit drop, (L, V)
    score = gain[pos_t].masked_fill(~allowed, float("-inf")).max(1).values   # (P,)
    top = score.topk(min(m, len(pos))).indices.tolist()
    return [pos[t] for t in top]

def gen_candidates(url, positions, layout):
    # make 1-char edits (substitute / delete / insert) at the chosen positions
    n, ps, hr = layout
    cands = set()
    for i in positions:
        if i >= n:  # append
            for c in ins_chars(n, layout):
                cands.add(url + c)
            continue
        chars = HOST_CHARS if (hr and hr[0] <= i < hr[1]) else PATH_CHARS
        for c in chars:
            if c != url[i]:
                cands.add(url[:i] + c + url[i + 1:])
        cands.add(url[:i] + url[i + 1:])  # delete
        for j in (i, i + 1):              # insert before i / after i
            ic = ins_chars(j, layout)
            if ic:
                for c in ic:
                    cands.add(url[:j] + c + url[j:])
    cands.discard(url)
    return list(cands)

# Attack
- up to 3 edits
- stop if predicted P(phish) is < threshold (successfully flipped prediction)
- get layout of url and top position to edit, then generate candidates
- score all candidates in 1 batch
- Sort by p_phish from lowest, stop if none beats the current value, and otherwise take the first candidate that passes `invalid_reason`
- Move to that candidate and repeat.


In [37]:
def attack_url(orig, p_clean):
    url, p_cur = orig, float(p_clean) #original url , model's predicted prob

    for _ in range(K):
        if p_cur < THRESHOLD:
            break
        layout = url_layout(url)
        positions = top_positions(url, layout)
        if not positions:
            break
        cands = gen_candidates(url, positions, layout)
        if not cands:
            break
        probs = p_phish_batch(cands) #run every candidate url through cnn

        best = None
        for j in np.argsort(probs):
            if probs[j] >= p_cur:            # no candidate left that lowers p_phish
                break
            if invalid_reason(orig, cands[j], K) == "": #candidate is valid
                best = j
                break
        if best is None:
            break
        url, p_cur = cands[best], float(probs[best])

    return {
        "orig_URL": orig,
        "URL": url,
        "label": 0,                           # dataset convention: 0 = phishing
        "p_clean": float(p_clean),
        "p_adv": p_cur,
        "n_edits": edit_distance(orig, url),
        "flipped": p_cur < THRESHOLD,
    }

## Smoke test

In [39]:
sample = test_df[test_df["label"] == 0].head(10)
p_s = p_phish_batch(sample["URL"].tolist()) # probabilities
sample = sample[p_s >= THRESHOLD].head(15) # edit urls predicted as phishing
p_s = p_phish_batch(sample["URL"].tolist())

for u, p in zip(sample["URL"], p_s):
    r = attack_url(u, p)
    print(f"{r['n_edits']} edits | {r['p_clean']:.3f} -> {r['p_adv']:.3f} | {r['orig_URL']}  =>  {r['URL']}")

3 edits | 0.997 -> 0.901 | liuy-9a930.web.app  =>  liuys9a93aa.web.app
3 edits | 0.997 -> 0.777 | mechinchem-5cb8a.web.app  =>  mechinches-5vh8a.web.app
3 edits | 0.981 -> 0.802 | fb-restriction-case-97be5.web.app  =>  fb-restriction-ciasev97be5o.web.app
3 edits | 0.995 -> 0.771 | metafb-tvz6efk.web.app  =>  metabb-tvp6efo.web.app
3 edits | 0.992 -> 0.916 | ggsexpole.web.app  =>  gugse.poe.web.app
3 edits | 1.000 -> 0.999 | helpadcase1002405739503874569.web.app  =>  helpadcase1002405e3950a87i569.web.app
3 edits | 0.982 -> 0.239 | jerode.ga  =>  jerode.ga?.i
3 edits | 0.996 -> 0.942 | perte-dimension.web.app  =>  perre-dtmensison.web.app


# Validation

In [ ]:
# call validation fns

# Summary

In [ ]:
# save in results
# add plots

# Save augmented dataset
- original + adversarial (satisfying constraints)

In [ ]:
def append_adversarial(original_df, adv):
    extra = adv.loc[adv["n_edits"] > 0, ["URL", "label"]]  # label stays 0 = phishing (dataset convention)
    return pd.concat([original_df[["URL", "label"]], extra], ignore_index=True)


train_aug = append_adversarial(train_df, train_adv)   
test_aug = append_adversarial(test_df, test_adv)

train_aug.to_csv(ADV_DIR / "cnn_train_augmented.csv", index=False)
test_aug.to_csv(ADV_DIR / "cnn_test_augmented.csv", index=False)
print("train:", len(train_df), "->", len(train_aug), "| test:", len(test_df), "->", len(test_aug))